In [ ]:
import random, os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'  # 必须放在导入其他库之前
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from matplotlib import pyplot as plt
from datetime import *
from real2D_Env import Env

In [ ]:
import gymnasium as gym

from stable_baselines3 import PPO

env = gym.make("CartPole-v1", render_mode="human")

model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=10_000)

vec_env = model.get_env()
obs = vec_env.reset()
for i in range(1000):
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, done, info = vec_env.step(action)
    vec_env.render()
    # VecEnv resets automatically
    # if done:
    #   obs = env.reset()

env.close()

In [ ]:
import gymnasium as gym
from my_USV_Env import MyUsvEnv_Gym
# Register the environment so we can create it with gym.make()
gym.register(
    id="myenv/MyUsvEnv_Gym-v0",
    entry_point=MyUsvEnv_Gym,
)
# 2. 创建环境实例
env = gym.make("myenv/MyUsvEnv_Gym-v0")  # 使用注册的id创建环境
from gymnasium.utils.env_checker import check_env
# This will catch many common issues
try:
    check_env(env)
    print("Environment passes all checks!")
except Exception as e:
    print(f"Environment has issues: {e}")

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
# Parallel environments
vec_env = make_vec_env("myenv/MyUsvEnv_Gym-v0", n_envs=4)
model = PPO("MultiInputPolicy", vec_env, verbose=1)
model.learn(total_timesteps=25000)
model.save("ppo_MyUsvEnv")

In [ ]:
# 创建测试环境（使用与训练相同的环境参数）
from stable_baselines3 import PPO
import gymnasium as gym
import matplotlib.pyplot as plt

model = PPO.load("ppo_SeaEnv")

# Register the environment so we can create it with gym.make()
gym.register(
    id="myenv/SeaEnv-v0",
    entry_point=SeaEnv,
)
env = gym.make("myenv/SeaEnv-v0")

episode_rewards = []
for _ in range(10):
    # 显式解包观测值和info字典
    obs, info = env.reset()  # 新版Gym返回元组
    terminated = False
    truncated = False
    total_reward = 0
    
    while not (terminated or truncated):
        action, _ = model.predict(obs, deterministic=True)  # 确保传入的是numpy数组
        # 新版Gym返回5个值
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
    
    episode_rewards.append(total_reward)

plt.plot(episode_rewards)
plt.show()
